# 03. Lexical, Semantic & Hybrid Search

In this notebook, we explore and compare the candidate retrieval stages:
1. **Lexical Retrieval (TF-IDF)**: Keyword baseline with n-grams and sublinear TF.
2. **Dense Semantic Retrieval (Sentence Transformers + FAISS)**: Embeddings generated using `all-MiniLM-L6-v2` and searched via Inner-Product index on L2-normalized vectors.
3. **Hybrid Retrieval**: Convex combination $\alpha \cdot \text{Semantic} + (1-\alpha) \cdot \text{TF-IDF}$ with structured marketplace filters.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

import time
import pandas as pd
import numpy as np
import config
from src.search import AirbnbSearchEngine, build_index

# Check if index is built, otherwise build it
if not config.FAISS_INDEX_PATH.exists():
    print("Building Search Index (TF-IDF + FAISS embeddings)... This runs once.")
    build_index()

# Initialize and load search engine
engine = AirbnbSearchEngine()
engine.load()
print(f"Loaded Search Engine with {len(engine.listings):,} indexed listings.")

## 1. Test Query Comparison: Lexical vs. Semantic Retrieval
Let's test an intent-driven natural language query:
`"quiet sunny apartment in Amsterdam with workspace and high speed wifi"`

In [ ]:
test_query = "quiet sunny apartment in Amsterdam with workspace and high speed wifi"

# TF-IDF
t0 = time.perf_counter()
tfidf_res = engine.tfidf_search(test_query, top_k=5)
t_tfidf = (time.perf_counter() - t0) * 1000

# Semantic
t0 = time.perf_counter()
semantic_res = engine.semantic_search(test_query, top_k=5)
t_sem = (time.perf_counter() - t0) * 1000

print(f"TF-IDF Search Latency: {t_tfidf:.1f} ms")
display(tfidf_res[['id', 'city', 'name', 'price_usd', 'review_scores_rating', 'tfidf_score']])

print(f"\nSemantic Search Latency: {t_sem:.1f} ms")
display(semantic_res[['id', 'city', 'name', 'price_usd', 'review_scores_rating', 'semantic_score']])

## 2. Hybrid Search with Structured Marketplace Filters

In [ ]:
t0 = time.perf_counter()
hybrid_res = engine.hybrid_search(
    query="cosy apartment with kitchen near city centre",
    top_k=5,
    alpha=0.6,
    city_filter=["Amsterdam"],
    max_price=200,
    min_rating=4.5
)
t_hybrid = (time.perf_counter() - t0) * 1000

print(f"Hybrid Search Latency: {t_hybrid:.1f} ms")
hybrid_res[['id', 'city', 'name', 'price_usd', 'review_scores_rating', 'hybrid_score', 'semantic_score', 'tfidf_score']]